In [1]:
import cv2
import dlib
import os
import pandas as pd
import glob
import re
import concurrent.futures

In [6]:
# Paths
base_dir = "../../Video_Data"
os.makedirs("Facial_Landmarks", exist_ok=True)
output_dir = "Facial_Landmarks" 

In [ ]:

# Initialize dlib's face detector and landmark predictor
face_detector = dlib.get_frontal_face_detector()
landmark_predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

def process_ir_videos(base_dir, output_dir):
    # Iterate over each person's folder (A to S)
    for person in os.listdir(base_dir):
        person_folder = os.path.join(base_dir, person)
        if not os.path.isdir(person_folder):
            continue  # Skip if not a folder

        print(f"Processing data for person: {person}")  

        # Look for Alert (A) and Drowsy (D) subfolders
        for session in ['A', 'D']:
            session_folder = os.path.join(person_folder, session)
            if not os.path.isdir(session_folder):
                print(f"Skipping {session} session for {person}: Folder not found.")
                continue  # Skip if session folder doesn't exist
            
            # Find IR videos in the session folder
            ir_video_paths = glob.glob(os.path.join(session_folder, f"{person}_IR_{session}.mp4"))
            if not ir_video_paths:
                print(f"No IR videos found for {person} in {session} session.")
                continue  # Skip if no matching videos
            
            # Create corresponding output folder
            output_folder = os.path.join(output_dir, person, session)
            os.makedirs(output_folder, exist_ok=True)

            # Process each video (should be only one per session)
            for ir_video_path in ir_video_paths:
                print(f"Processing video: {os.path.basename(ir_video_path)}")

                video_capture = cv2.VideoCapture(ir_video_path)
                frame_count = 0

                while True:
                    ret, frame = video_capture.read()
                    if not ret:
                        break  # End of video

                    frame_count += 1
                    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  # Convert to grayscale

                    # Detect faces in the frame
                    faces = face_detector(gray_frame)

                    for i, face in enumerate(faces):  # Loop through detected faces
                        # Predict landmarks
                        landmarks = landmark_predictor(gray_frame, face)

                        # Extract landmark coordinates
                        landmarks_list = [(landmarks.part(n).x, landmarks.part(n).y) for n in range(68)]

                        # Save landmarks as CSV with new naming format
                        output_filename = f"{person}_FL_{session}_frame{frame_count}.csv"
                        output_path = os.path.join(output_folder, output_filename)
                        pd.DataFrame(landmarks_list, columns=["X", "Y"]).to_csv(output_path, index=False)

                video_capture.release()
                print(f"✅ Landmarks extracted and saved for {os.path.basename(ir_video_path)}")

    print("✅ Processing completed.")

# Run the function
process_ir_videos(base_dir, output_dir)

##### Concatenate ######

In [ ]:
# Path to the root directory containing all user folders
root_folder = "Facial_Landmarks"
output_parent_dir = "Extracted_Features"
os.makedirs(output_parent_dir, exist_ok=True)  # Ensure output directory exists

# Regular expression to extract frame numbers from filenames like "A_FL_A_frame1.csv"
frame_number_pattern = re.compile(r'frame(\d+)')

def process_file(file_name, input_folder):
    """Read a CSV file and return the flattened data along with the frame number."""
    file_path = os.path.join(input_folder, file_name)

    # Extract frame number from filename
    frame_match = frame_number_pattern.search(file_name)
    if not frame_match:
        print(f"Skipping file {file_name}: No valid frame number found.")
        return None  # Skip files without a valid frame number
    
    frame_number = int(frame_match.group(1))

    try:
        # Read the CSV file
        data = pd.read_csv(file_path)

        # Flatten the data into a single row
        flattened_row = data.values.flatten()

        # Create column names dynamically
        num_points = len(flattened_row) // 2  # Assuming X, Y pairs
        column_names = [f"{axis}{i+1}" for i in range(num_points) for axis in ['X', 'Y']]

        return flattened_row, column_names, frame_number

    except Exception as e:
        print(f"Error processing {file_name}: {e}")
        return None

# Process all user folders
for user_folder in os.listdir(root_folder):
    user_folder_path = os.path.join(root_folder, user_folder)
    if not os.path.isdir(user_folder_path):
        continue  # Skip non-directory files

    print(f"Processing user: {user_folder}")

    # Process each session (e.g., A or D)
    for session in os.listdir(user_folder_path):
        session_folder_path = os.path.join(user_folder_path, session)
        if not os.path.isdir(session_folder_path):
            continue  # Skip non-directory files

        print(f"  Processing session: {session}")

        # Get a list of all CSV files for the current user and session
        files = [f for f in os.listdir(session_folder_path) if f.endswith(".csv")]
        
        # Sort only if valid frame numbers are found
        try:
            files_sorted = sorted(files, key=lambda x: int(frame_number_pattern.search(x).group(1)) if frame_number_pattern.search(x) else float('inf'))
        except Exception as e:
            print(f"Sorting error in {user_folder} session {session}: {e}")
            continue  # Skip this session if sorting fails

        if not files_sorted:
            print(f"  No valid CSV files found in {session_folder_path}. Skipping...")
            continue  # Skip if no valid files

        # Use concurrent futures to read and process files in parallel
        with concurrent.futures.ThreadPoolExecutor() as executor:
            results = list(filter(None, executor.map(lambda file: process_file(file, session_folder_path), files_sorted)))

        if not results:
            print(f"  No valid data processed for {user_folder} session {session}. Skipping...")
            continue

        # Extract the data and frame numbers
        data_rows = [result[0] for result in results if result]
        frame_numbers = [result[2] for result in results if result]

        if not data_rows:
            print(f"  No valid data rows for {user_folder} session {session}. Skipping...")
            continue

        # Create DataFrame with the landmark data
        all_frames_data = pd.DataFrame(data_rows, columns=results[0][1])

        # Add frame numbers as the first column
        all_frames_data.insert(0, 'Frame_Number', frame_numbers)

        # Create session output folder inside Extracted_Features
        user_output_folder = os.path.join(output_parent_dir, user_folder, session)
        os.makedirs(user_output_folder, exist_ok=True)

        # Generate the correct file name format: "User_FL_Session.csv"
        output_filename = f"{user_folder}_FL_{session}.csv"
        output_csv = os.path.join(user_output_folder, output_filename)

        # Save the final FL file in the correct user/session folder
        all_frames_data.to_csv(output_csv, index=False)

        print(f"✅ Combined {output_filename} saved to {output_csv}")
